Prendete un grafo di 4-5 nodi, con almeno un nodo senza archi uscenti (dangling) e un ciclo.

In [2]:
import numpy as np

# Directed edge (i, j): a link from page i to page j. (i-->j) 
# Direction matters: (0, 2) and (2, 0) are two different edges.
#  edges i -> j (nodes 0..4); node 4 has no outgoing links (dangling)

edges = [(0, 1), (0, 2), (1, 2), (2, 0), (3, 2), (3, 4)]

n = 5                        # number of nodes

A = np.zeros((n,n))          ## Adiancy matrix nxn --> it describes the graph. --> it says if the link exist or not
print(A)

for i, j in edges:
    A[j, i] = 1              # column i holds the out-links of node i
print("\n Position of the edges to A: \n",A)

out = A.sum(axis=0)          # Out-degree of node i: number of edges leaving i (links on page i).
                             # For the example graph: out = [2, 1, 1, 2, 0]  

                             # the random surfer picks each outgoing link with equal probability.
                             # A NODE with OUT-DEGREE 0 is a DANGLING NODE --> its column stays all zeros
                             # and is later replaced by the teleportation vector v.
 
# Boolean mask marking dangling nodes (nodes with no outgoing links, like the last column, the n°4)
mk_dangling = (out == 0) # [False, False, False, False, True]

# Creation of Transition matrix P: is built by dividing each column of A by its out-degree so it sums to 1.
## Dangling columns remains all zeros (division is skipped where out == 0). 

## np.divide(A, out) divide A for the vect out. out it's allined to the columns of A, so column i of A is divided for out[i]
# where=out > 0 says that it performs the division only where=out > 0 so 1,2 ecc. not 0
# out=np.zeros_like(A) it's the array where the result division is written
P = np.divide(A, out, out=np.zeros_like(A), where=out > 0)

print("\n Transition matrix P: \n",P)

# Damping factor: probability of following a link (1 - alpha = teleportation that means the ignore the link and jumps to another page).
# It guarantees a unique positive PageRank vector and controls both the convergence rate (|lambda_2| <= alpha) and the sensitivity (~ 1/(1 - alpha)).
alpha = 0.85

# Teleportation vector: uniform over the n nodes. It's a distribution prob that a surfer choose the arrive page.
# n components that sum up to 1. It's uniform so 1/n for each page is equally probable.
# with another v we create a personalized PageRank
v = np.ones(n) / n
print("\n Teleportation vect v:",v)

# We want P column-stochastic matrix
# np.outer(v, dangling) built a 5x5 matrix where the i-column is equal to v if the i-node id dangling  othewise is all zero
P_fixed = P + np.outer(v, mk_dangling)

print("\n Fixed_Transition matrix P (it sums up to 1): \n",P_fixed)


# Now i've to build the Google Matrix --> with probability alpha the surfer follows the links of P_fixed,
# with probability 1 - alpha it teleports according to v. # np.outer(v, np.ones(n)) is the RANK-ONE matrix v*e^T (every column equals v).
# v e^T = [ v  v  v  v  v ]    (5 columns, every one is v) ; v is column vect nx1 and e^T is 1xn
teleport_matrix = np.outer(v, np.ones(n))
print("\n v*e^T",teleport_matrix)
G = alpha * P_fixed + (1 - alpha) * teleport_matrix

print("\nGoogle Matrix:\n",G)

##  G must be column-stochastic (each column sums to 1).
print("\nColumn sums of G:", G.sum(axis=0))
assert np.allclose(G.sum(axis=0), 1)


# Reference PageRank (dense eigen-decomposition, only for small tests):
# eigenvector of G for the eigenvalue 1, made positive and normalized to sum 1.
w, V = np.linalg.eig(G)        # calculate eigenvalues
k = np.argmax(w.real)          # index of the dominant eigenvalue
x_ref = np.abs(V[:, k].real)   # eigenvectors are defined up to sign
x_ref /= x_ref.sum()           # normalize so the entries sum to 1

print("\nDominant eigenvalue:", w.real[k]) # 0.9999999999999987
print("PageRank (reference):", x_ref) # [0.35017836 0.1884167  0.36539702 0.03959089 0.05641702]


[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]

 Position of the edges to A: 
 [[0. 0. 1. 0. 0.]
 [1. 0. 0. 0. 0.]
 [1. 1. 0. 1. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0.]]

 Transition matrix P: 
 [[0.  0.  1.  0.  0. ]
 [0.5 0.  0.  0.  0. ]
 [0.5 1.  0.  0.5 0. ]
 [0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.5 0. ]]

 Teleportation vect v: [0.2 0.2 0.2 0.2 0.2]

 Fixed_Transition matrix P (it sums up to 1): 
 [[0.  0.  1.  0.  0.2]
 [0.5 0.  0.  0.  0.2]
 [0.5 1.  0.  0.5 0.2]
 [0.  0.  0.  0.  0.2]
 [0.  0.  0.  0.5 0.2]]

 v*e^T [[0.2 0.2 0.2 0.2 0.2]
 [0.2 0.2 0.2 0.2 0.2]
 [0.2 0.2 0.2 0.2 0.2]
 [0.2 0.2 0.2 0.2 0.2]
 [0.2 0.2 0.2 0.2 0.2]]

Google Matrix:
 [[0.03  0.03  0.88  0.03  0.2  ]
 [0.455 0.03  0.03  0.03  0.2  ]
 [0.455 0.88  0.03  0.455 0.2  ]
 [0.03  0.03  0.03  0.03  0.2  ]
 [0.03  0.03  0.03  0.455 0.2  ]]

Column sums of G: [1. 1. 1. 1. 1.]

Dominant eigenvalue: 0.9999999999999987
PageRank (reference): [0.35017836 0.1884167  0.365

In [3]:
import numpy as np
import scipy.sparse as sp


# Define a function that build the adjacency matrix --> it takes the list of edges of the graph and turn it into A
def build_adjacency(edges, n):

    # edges: list of directed edges (i, j), a link from page i to page j, and n: number of nodes
    # Returns the SPARSE adjacency matrix A with A[j, i] = 1 for each edge i -> j.

    # Step 1: keep each edge Only once and ignore self-loops [like (2,2) or (1,1) ]
    clean_edges = set()                 # a set() CANNOT contain the SAME ELEMENT twice
    for edge in edges:
        i = edge[0]                     # source: where the link starts --> i
        j = edge[1]                     # destination: where the link ends/arrives --> j
        if i != j:                      # skip self-loops (page linking to itself)
            clean_edges.add((i, j))     # adding an edge already present does nothing

    # Step 2: SPLIT the edges (without self-loops, ) into 2 lists, sources and destinations.
    src = []                            # SOURCE of each edge  -> column index of A
    dst = []                            # DESTINATION of each edge -> row index of A
    for edge in clean_edges:
        i = edge[0]
        j = edge[1]
        src.append(i)
        dst.append(j)

    # Step 3: one "1" for every edge. --> it creates a vector of 1 for every edges, could be every length
    data = np.ones(len(clean_edges))

    # Step 4: csr_matrix build the sparse matrix. Entry (row, column) = (dst, src), so column i holds the out-links of node i.
    # shape=(n, n) --> dimension; data are the values to put in the A matrix;  (dst, src) is a tuple that contains 2 lists
    A = sp.csr_matrix((data, (dst, src)), shape=(n, n))


    return A

In [27]:
# RANK SINK

#  Nodes 0, 1, 2 form the "main" part; nodes 3 and 4 form the sink.
# The sink only links to itself: 3 -> 4 and 4 -> 3. There is one way in: 2 -> 3.

edges_sink = [(0, 1), (1, 2), (2, 0), (2, 3), (3, 4), (4, 3)]
n_sink = 5                                   # number of nodes (not edges)
A_sink = build_adjacency(edges_sink, n_sink)

A = A_sink.toarray()  # Save A_sink in A ## is a dense copy only for the test --> '.toarray' coverts it in a dense matrix (simple array nxn)
print("\nA_sink as a dense array:\n", A)

out = A.sum(axis=0)
print("\nout:\n",out)

P = np.divide(A, out, out=np.zeros_like(A), where=out > 0)
print("\nP:\n",P)



A_sink as a dense array:
 [[0. 0. 1. 0. 0.]
 [1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 1.]
 [0. 0. 0. 1. 0.]]

out:
 [1. 1. 2. 1. 1.]

P:
 [[0.  0.  0.5 0.  0. ]
 [1.  0.  0.  0.  0. ]
 [0.  1.  0.  0.  0. ]
 [0.  0.  0.5 0.  1. ]
 [0.  0.  0.  1.  0. ]]


### <mark> Why convert a sparse matrix with .toarray()

A_sink is a sparse matrix (csr_matrix). A = A_sink.toarray() returns a dense NumPy copy (n × n, with all the zeros stored), used in tests on small graphs.

1. The experiment code uses dense-array operations

Lines such as np.zeros_like(A), np.divide(A, out, ...) and np.outer(...) are written for NumPy arrays. With a sparse matrix, some of these functions do not behave as you would expect: they may raise errors or give different results. Converting first avoids the problem.

2. To see the matrix

If you print a sparse matrix, Python only shows the list of occupied positions (row, column, value). With .toarray() you see it as a 5 × 5 table.

### <mark> Surfer WITHOUT teleportation<>

In [28]:
# Now we have to consider a surfer WITHOUT teleportation, that is alpha = 1 --> G = 1 * P + 0 * v e^T = P.
# With alpha = 1 the Google matrix is G = P_fixed. This graph has NO dangling
# nodes, so P_fixed = P and we can use P directly.

# Start from the uniform distribution: 1/n for each of the n nodes.
x = np.ones(n_sink) / n_sink
print("start:", x, "\n")

# Now apply P repeatedly. x[i] = probability that the surfer is on page i.
for k in range(1, 201):
    x = P @ x                                    # one step of the random surfer
    if k in (1, 3, 10, 50, 199, 200):
        print(k, np.round(x, 4), round(x.sum(), 6))   # the total probability must stay 1

start: [0.2 0.2 0.2 0.2 0.2] 

1 [0.1 0.2 0.2 0.3 0.2] 1.0
3 [0.1 0.1 0.1 0.4 0.3] 1.0
10 [0.0125 0.025  0.025  0.4375 0.5   ] 1.0
50 [0.     0.     0.     0.4667 0.5333] 1.0
199 [0.     0.     0.     0.5333 0.4667] 1.0
200 [0.     0.     0.     0.4667 0.5333] 1.0


### <mark> Observation: Surfer without teleportation (rank sink graph)

1. **The probability leaves nodes 0, 1, 2.** It goes from 0.1-0.2 to 0.0125 at step 10 and to 0 at step 50. Every time the surfer goes through node 2, half of the probability takes the link to node 3 and enters the sink. It never leaves the sink, so the main part empties out.

2. **All the probability ends up in nodes 3 and 4.** At step 50 the values are 0.4667 and 0.5333, which still sum to 1: the probability does not disappear, it accumulates in the sink.

3. **But nodes 3 and 4 do not settle.** Compare rows 199 and 200: at 199 we have `0.5333 0.4667`, at 200 we have `0.4667 0.5333`. The values swap at every step, because the sink is a cycle of length 2 (3 → 4 → 3) and the probability bounces back and forth.

### <mark> Surfer with teleportation

In [ ]:
# Now we consider a surfer WITH teleportation, still with a uniform
# teleportation vector: 1/n for each of the n nodes.

alpha = 0.85
v = np.ones(n_sink) / n_sink                       # teleportation vector

# Build the Google matrix: G = alpha * P + (1 - alpha) * v e^T
v_eT = np.outer(v, np.ones(n_sink))                # rank-one matrix v e^T
G = alpha * P + (1 - alpha) * v_eT                 # no dangling nodes here, so P is already stochastic

# Same iteration, now with G
x = np.ones(n_sink) / n_sink
for k in range(1, 201):
    x = G @ x
    if k in (1, 3, 10, 50, 199, 200):
        total = x.sum()                            # total probability: must stay equal to 1
        print(k, np.round(x, 4), round(total, 6))



# Check against the reference PageRank (dense eigen-decomposition, small tests only)
w, V = np.linalg.eig(G)
i = np.argmax(w.real)
x_ref = np.abs(V[:, i].real)
x_ref /= x_ref.sum()
print("\neig:      ", np.round(x_ref, 4))
print("iteration:", np.round(x, 4))
print("same result?", np.allclose(x, x_ref))

1 [0.115 0.2   0.2   0.285 0.2  ] 1.0
3 [0.115  0.1278 0.1386 0.3464 0.2723] 1.0
10 [0.0784 0.0988 0.1139 0.3613 0.3476] 1.0
50 [0.0773 0.0957 0.1114 0.3706 0.345 ] 1.0
199 [0.0773 0.0957 0.1114 0.3706 0.345 ] 1.0
200 [0.0773 0.0957 0.1114 0.3706 0.345 ] 1.0

eig:       [0.0773 0.0957 0.1114 0.3706 0.345 ]
iteration: [0.0773 0.0957 0.1114 0.3706 0.345 ]
same result? True


#### <mark>Rank sink (ITA)

Un sink (o rank sink) è un gruppo di pagine da cui il navigatore, una volta entrato, non può più uscire, perché i link di quelle pagine puntano solo ad altre pagine dello stesso gruppo.

Nel nostro grafo è il gruppo {3, 4}:

il nodo 3 linka solo al nodo 4,
il nodo 4 linka solo al nodo 3,
nessuno dei due ha un link verso 0, 1 o 2.

L'unico modo di entrarci è il link 2 → 3. Una volta dentro, il navigatore rimbalza tra 3 e 4 per sempre.

Si chiama "sink" (pozzo, scarico) perché raccoglie probabilità: a ogni passaggio dal nodo 2 metà della probabilità entra, e nessuna ne esce. Nell'esperimento senza teletrasporto la probabilità dei nodi 0, 1, 2 andava a zero e tutto finiva in 3 e 4.

Differenza con un dangling node

Anche il dangling node è una trappola, ma in un altro modo:

Il dangling node non ha nessun link in uscita (il navigatore non sa dove andare, e la colonna di P è tutta zero).
Il rank sink ha link in uscita, ma tutti restano dentro il gruppo. Le colonne di P sono normali e sommano a 1, quindi P è stocastica, ma il grafo non è fortemente connesso.
Perché il teletrasporto aiuta

Con probabilità 1 - alpha il navigatore ignora i link e salta a una pagina qualsiasi, quindi può sempre uscire dal sink. Per questo nel secondo esperimento il vettore converge e anche i nodi 0, 1, 2 hanno probabilità positiva. Il sink resta comunque il gruppo con più PageRank, circa il 72%, perché i nodi che si linkano solo tra loro si "restituiscono" a vicenda la probabilità.

#### <mark>Rank sink (ENG)


A sink (or rank sink) is a group of pages that the surfer cannot leave once inside, because the links of those pages only point to other pages of the same group.

In our graph it is the group {3, 4}:

node 3 only links to node 4,
node 4 only links to node 3,
neither of them has a link to 0, 1 or 2.

The only way in is the link 2 → 3. Once inside, the surfer bounces between 3 and 4 forever.

It is called a "sink" because it collects probability: every time the surfer goes through node 2, half of the probability enters, and none of it leaves. In the experiment without teleportation the probability of nodes 0, 1, 2 went to zero and everything ended up in 3 and 4.

Difference from a dangling node

A dangling node is also a trap, but in a different way:

A dangling node has no outgoing links at all (the surfer does not know where to go, and the column of P is all zeros).
A rank sink does have outgoing links, but all of them stay inside the group. The columns of P are normal and sum to 1, so P is stochastic, but the graph is not strongly connected.
Why teleportation helps

With probability 1 - alpha the surfer ignores the links and jumps to any page, so it can always leave the sink. This is why in the second experiment the vector converges and nodes 0, 1, 2 also get positive probability. The sink still remains the group with the most PageRank, about 72%, because nodes that only link to each other "give back" the probability to one another.

In [ ]:
# COMMUNITIES
# Community 1: nodes 0, 1, 2. Community 2: nodes 3, 4, 5.
# Inside each community every node links to the other two nodes (dense block).
# Between the 2 communities there is ONLY ONE link in each direction.

edges_comm = [
    (0, 1), (0, 2), (1, 0), (1, 2), (2, 0), (2, 1),      # community 1: nodes 0, 1, 2
    (3, 4), (3, 5), (4, 3), (4, 5), (5, 3), (5, 4),      # community 2: nodes 3, 4, 5
    (2, 3), (5, 0),                                      # the few links between communities
]

# How to read the incoming links of a node. --> An incoming link to node 0 is an edge that ends in 0, i.e. with 0 as second number.  In edges_comm:
#   (1, 0)  -> from node 1
#   (2, 0)  -> from node 2
#   (5, 0)  -> from node 5
# so node 0 has 3 incoming links. In A, row j = 0 is [0, 1, 1, 0, 0, 1]: the 1s are in columns 1, 2, 5, i.e. the nodes that link to node 0.
# Same for node 1: (0, 1) and (2, 1), so it receives from nodes 0 and 2.

n_comm = 6                                   # number of nodes (not edges)
A_comm = build_adjacency(edges_comm, n_comm)
A = A_comm.toarray()                         # dense copy, only for this small test
print("\nA_comm:\n", A)

out = A.sum(axis=0)                          # out-degrees: links leaving each node, i.e. in how many parts
print("\nout:\n", out)                       # each node splits its PageRank

# Nodes 2 and 5 have out-degree 3 because, besides the 2 links inside their own community, each has one link to the other community: (2, 3) and (5, 0).

# Transition matrix: P[j, i] = 1 / out[i] IF there is an edge i -> j, so each row of P
# contains the coefficients 1/2 or 1/3 that each node passes to the nodes it links.

P = np.divide(A, out, out=np.zeros_like(A), where=out > 0)
print("\nP:\n", P, "\n")                            # No out-degree is 0, so there are no dangling nodes: P is already column-stochastic (every column sums to 1).

alpha = 0.85                                                    # Damping factor
v = np.ones(n_comm) / n_comm                                    # uniform teleportation vector
G = alpha * P + (1 - alpha) * np.outer(v, np.ones(n_comm))      # P is already stochastic (no dangling nodes)

# Surfer WITH teleportation: apply G repeatedly from the uniform distribution
x = np.ones(n_comm) / n_comm
for k in range(1, 201):
    x = G @ x
    if k in (1, 3, 10, 50, 199, 200):
        print(k, np.round(x, 4), round(x.sum(), 6))




# Check against the reference PageRank (dense eigen-decomposition, small tests only)
# np.linalg.eig(G) calculates all eigenvalues and eigenvector of the matrix G --> it works only for small graphs 

w, V = np.linalg.eig(G)                     # w: the n eigenvalues of G. V: a matrix whose column k is the eigenvector of w[k].
i = np.argmax(w.real)                       # Index of the largest eig_val. For G it is the eigenvalue 1, which is the one whose eigenvector is the PageRank.

x_ref = np.abs(V[:, i].real)                # # V[:, i] is the column i, i.e. the eigenvector of that eigenvalue. .real drops the (zero) imaginary part. 
                                            #np.abs fixes the sign: an eigenvector is defined up to a factor, so it can come out with all negative entries.

x_ref /= x_ref.sum()                        # eig returns a vect of length 1 in the 2-norm, NOT with sum 1. Dividing by the sum makes the entries sum to 1, so it is a probability distribution like our x.

print("\neig:      ", np.round(x_ref, 4))
print("iteration:", np.round(x, 4))
print("same result?", np.allclose(x, x_ref))


A_comm:
 [[0. 1. 1. 0. 0. 1.]
 [1. 0. 1. 0. 0. 0.]
 [1. 1. 0. 0. 0. 0.]
 [0. 0. 1. 0. 1. 1.]
 [0. 0. 0. 1. 0. 1.]
 [0. 0. 0. 1. 1. 0.]]

out:
 [2. 2. 3. 2. 2. 3.]

P:
 [[0.         0.5        0.33333333 0.         0.         0.33333333]
 [0.5        0.         0.33333333 0.         0.         0.        ]
 [0.5        0.5        0.         0.         0.         0.        ]
 [0.         0.         0.33333333 0.         0.5        0.33333333]
 [0.         0.         0.         0.5        0.         0.33333333]
 [0.         0.         0.         0.5        0.5        0.        ]] 

1 [0.1903 0.1431 0.1667 0.1903 0.1431 0.1667] 1.0
3 [0.1845 0.1488 0.1667 0.1845 0.1488 0.1667] 1.0
10 [0.1832 0.1501 0.1667 0.1832 0.1501 0.1667] 1.0
50 [0.1832 0.1501 0.1667 0.1832 0.1501 0.1667] 1.0
199 [0.1832 0.1501 0.1667 0.1832 0.1501 0.1667] 1.0
200 [0.1832 0.1501 0.1667 0.1832 0.1501 0.1667] 1.0

eig:       [0.1832 0.1501 0.1667 0.1832 0.1501 0.1667]
iteration: [0.1832 0.1501 0.1667 0.1832 0.1501 0.166


### <mark> Observations

- The sum is still 1 at every step, so `G` is column-stochastic and the normalization is preserved.

- From step 10 to step 200 the probability vector does not change (to 4 decimals): it has stabilized after about 10 steps. In the rank sink case we needed about 50 steps, so the convergence is faster here.

- `x[0] = x[3]`, `x[1] = x[4]`, `x[2] = x[5]`, because the two communities are built in the same way.

- <mark>Nodes 0 and 3 have the highest PageRank)</mark> because they have <mark>3 incoming links</mark> (two from their own community and one from the other one).

- Nodes 1 and 4 have the lowest PageRank: each has 2 incoming links. --> Node 1 receives from nodes 0 and 2: <mark>(0, 1) and (2, 1)</mark>. --> Node 4 receives from nodes 3 and 5: <mark>(3, 4) and (5, 4)</mark>.

See `out = [2. 2. 3. 2. 2. 3.]`:
- For example, node 1 receives `x0/2 + x2/3`, while node 2 receives `x0/2 + x1/2`. The second sum is larger, because node 1 passes 1/2 of its PageRank while node 2 passes only 1/3. This is why node 2 ends up with a higher PageRank than node 1.

```py
print("row 1 of P:", np.round(P[1], 4))   # [0.5, 0, 0.3333, 0, 0, 0]  ->  x0/2 + x2/3
print("row 2 of P:", np.round(P[2], 4))   # [0.5, 0.5, 0, 0, 0, 0]     ->  x0/2 + x1/2
```

In [ ]:
### The function describe(A) allows us to know the graph before the use, it summarizes the information

from scipy.sparse.csgraph import connected_components   # put this import at the top of the notebook


def describe(A, name="graph"):
    # the matrix A is a sparse adjacency matrix nxn with A[j, i] = 1 for the edge i -> j.

    n = A.shape[0]                              # number of NODES (A is n x n) --> we take the first dimention
    n_edges = A.nnz                             # number of NON-zero entries = number of EDGES

    # Out-degrees: SUM of each column. For a sparse matrix .sum() returns NumPy "matrix" of shape (1, n); 
    # np.asarray(...).ravel() turns it into a simple 1-D vector of length n and it is what we need
    # like before they are the sum of outcoming/uscenti edges
    out_deg = np.asarray(A.sum(axis=0)).ravel() 

    # incoming edges or in-degrees
    in_deg = np.asarray(A.sum(axis=1)).ravel()

    #mask for finding the 0 in the out_degree and find if P has columns with 0
    mk_out = (out_deg == 0)
    mk_in = (in_deg == 0)

    n_dangling = int((mk_out).sum())      # number of nodes with NO outgoing links (dangling) (sum how many 0 are in the out_deg)
    n_no_in = int((mk_in).sum())          # number of nodes that nobody links to (NO incoming links)



    # Strongly connected components (scc): groups of nodes where every node can reach every other. 
    # labels[i] = index of the component that contains node i.

    n_scc, labels = connected_components(A, directed=True, connection="strong")  # It splits a graph into groups of nodes that are connected to each other, 
                                                                                 # and it tells you which group each node belongs to 

    sizes = np.bincount(labels)                                                  # number of nodes in each component, with bincount i can count how many times there is a number in a list
                                                                                 # for example [3, 1, 1] component 0 has 3 nodes, component 1 has 1, component 2 has 1
    sizes_list = sorted(sizes.tolist(), reverse=True)                            # .tolist() from vect Numpy to a Python list and reverse true is descent order -->  (largest component first)



    print("\nNodes:",n,
          "\nEdges:", n_edges,
          "\nDangling nodes (out-degree 0):", n_dangling,
          "\nNodes with NO incoming links:",n_no_in,
          "\nStrongly connected components (scc):", n_scc,
          "\nsizes:", sizes_list)



# labels = [1, 1, 1, 0, 2] reads as: node 0 is in component 1, node 1 in component 1,
# node 2 in component 1, node 3 in component 0, node 4 in component 2.
# bincount = [1, 3, 1] means: component 0 has 1 node, component 1 has 3 nodes,
# component 2 has 1 node.
# connection="strong": looks for strongly connected components, i.e. groups where every
# node can reach every other one following the arrows. With "weak" the direction of the
# arrows would be ignored, and any group connected in some way would count as one.

In [67]:
describe(build_adjacency(edges, 5), "small example")
describe(A_sink, "rank sink")
describe(A_comm, "communities")


Nodes: 5 
Edges: 6 
Dangling nodes (out-degree 0): 1 
Nodes with NO incoming links: 1 
Strongly connected components (scc): 3 
sizes: [3, 1, 1]

Nodes: 5 
Edges: 6 
Dangling nodes (out-degree 0): 0 
Nodes with NO incoming links: 0 
Strongly connected components (scc): 2 
sizes: [3, 2]

Nodes: 6 
Edges: 14 
Dangling nodes (out-degree 0): 0 
Nodes with NO incoming links: 0 
Strongly connected components (scc): 1 
sizes: [6]


### <mark> What describe(A) is for (ITA)


describe(A) serve a conoscere un grafo prima di usarlo: stampa un riassunto con le sue caratteristiche principali, così non si lavora "alla cieca". Ha tre usi nel progetto.

1. Controllare che il grafo sia quello giusto

Con i grafi sintetici si verifica che la struttura sia quella voluta (per esempio, nel rank sink, che nodi e archi siano quelli attesi). Con il grafo reale serve a vedere se l'import è andato bene: un numero di nodi o di archi sbagliato avvisa subito di un errore nel caricamento.

2. Sapere quali problemi può dare il PageRank

I numeri stampati dicono cosa aspettarsi:

I dangling nodes (out-degree 0) richiedono la correzione con v. Se sono tanti, come in web-Stanford, la correzione pesa molto sul risultato.
I nodi senza link in ingresso avranno il PageRank più basso possibile, cioè solo la quota del teletrasporto.
Le componenti fortemente connesse indicano se esistono gruppi chiusi come i rank sink, che rallentano la convergenza.

3. Descrivere i dati nella relazione

La traccia chiede "una descrizione dei dati o delle matrici di prova". Con describe(A) si ottengono in una riga i numeri da mettere in una tabella:

| Grafo | Nodi | Archi | Dangling nodes | Senza link in ingresso |
|---|---|---|---|---|
| Esempio piccolo | 5 | 6 | 1 | 1 |
| Rank sink | 5 | 6 | 0 | 0 |
| Comunità | 6 | 14 | 0 | 0 |
| Grafo reale | … | … | … | … |

<mark>Non è un algoritmo del PageRank</mark>: non calcola nulla di nuovo, ma dà il contesto per interpretare i risultati e per giustificare le scelte (per esempio, perché il grafo reale ha bisogno della correzione dei dangling nodes). Con centinaia di migliaia di nodi non si può più guardare la matrice a occhio: lì describe(A) è l'unico modo di capire com'è fatto il grafo.

### <mark> What describe(A) is for (ENG)


describe(A) is used to get to know a graph before using it: it prints a summary of its main features, so that we do not work "blind". It has three uses in the project.

1. Check that the graph is the right one

With synthetic graphs we check that the structure is the intended one (for example, in the rank sink, that nodes and edges are the expected ones). With the real graph it shows whether the import went well: a wrong number of nodes or edges immediately warns of a loading error.

2. Know which problems PageRank may run into

The printed numbers tell what to expect:

Dangling nodes (out-degree 0) require the correction with v. If there are many, as in web-Stanford, the correction weighs heavily on the result.
Nodes with no incoming links will get the lowest possible PageRank, that is, only the teleportation share.
Strongly connected components show whether closed groups such as rank sinks exist, which slow down convergence.

3. Describe the data in the report

The assignment asks for "a description of the data or test matrices". With describe(A) we get in one line the numbers to put in a table:

| Grafo | Nodi | Archi | Dangling nodes | Senza link in ingresso |
|---|---|---|---|---|
| Esempio piccolo | 5 | 6 | 1 | 1 |
| Rank sink | 5 | 6 | 0 | 0 |
| Comunità | 6 | 14 | 0 | 0 |
| Grafo reale | … | … | … | … |

It is not a PageRank algorithm: it does not compute anything new, but it gives the context to interpret the results and to justify our choices (for example, why the real graph needs the dangling-node correction). With hundreds of thousands of nodes we can no longer look at the matrix by eye: there, describe(A) is the only way to understand what the graph looks like.

In [ ]:
import os

path = os.path.expanduser("~/Desktop/archive/graph.json")     # adjust if your folder is elsewhere

print("size (GB):", round(os.path.getsize(path) / 1e9, 2))

with open(path, "r", encoding="utf-8") as f:
    print(f.read(500))                                         # first 500 characters only

size (GB): 2.85
{
  "nodes": [
    {
      "id": "Verstanclahorn"
    },
    {
      "id": "Universal Channel (Australia)"
    },
    {
      "id": "Collegium Da Vinci"
    },
    {
      "id": "Lake Mogan"
    },
    {
      "id": "Scenes from Goethe's Faust"
    },
    {
      "id": "Night of Champions (2010)"
    },
    {
      "id": "Camocuautla"
    },
    {
      "id": "Baima language"
    },
    {
      "id": "Mercury (2005 novel)"
    },
    {
      "id": "Kam Heskin"
    },
    {
      "id": "Andando nas Nuvens"
    },
    {
      "id": "Suppression of the Jesuits"
    },
    {
      "id": "Vosges département"
    },
    {
      "id": "1939 Green Bay Packers season"
    },
    {
      "id": "Sherri Tenpenny"
    },
    {
      "id": "Kids on Strike!"
    },
    {
      "id": "Mirror Lake Library"
    },
    {
      "id": "Gamepressure"
    },
    {
      "id": "Energy wasting"
    },
    {
      "id": "Jhonny Acosta"
    },
    {
      "id": "Arab nationalism"
    },
    {
   